# 06. AI Brain Surgery: Knowledge Editing (ROME, MEMIT), Stitching, and Pruning

- **Category:** 10_xai_and_ai_analysis_methods
- **Lineage & Citations:** Lenc & Vedaldi (2015 *Understanding image representations by measuring their equivariance and equivalence: Model Stitching*); Csiszárik et al. (2021 *Similarity and Matching of Neural Network Representations*); Meng et al. (2022 *Locating and Editing Factual Associations in GPT: ROME*); Meng et al. (2023 *Mass-Editing Memory in a Transformer: MEMIT*).
- **Purpose:** Implement closed-form knowledge editing via Rank-One Model Editing (ROME), multi-fact mass editing (MEMIT), linear Model Stitching, and surgical neuron pruning.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [05. Mechanistic Interpretability Probing Lens Patching SAE](05_mechanistic_interpretability_probing_lens_patching_and_sae.ipynb) $\to$ `🔗 Downstream Syntheses:` [07. Representation Similarity Metrics CKA SVCCA RSA](07_ai_analysis_representation_similarity_cka_svcca_rsa_and_intrinsic_dim.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not perform ROME or MEMIT edits with singular or poorly estimated key covariance matrices $C$ (causes severe knowledge bleed and model corruption); avoid stitching across incompatible representational regimes without checking CKA similarity.

## Mathematical Foundations & Governing Equations

1. **ROME Rank-One Update**: $\Delta W = (v_* - W k_*) \frac{k_*^T C^{-1}}{k_*^T C^{-1} k_*}, \quad (W + \Delta W) k_* = v_*$
2. **MEMIT Mass Edit**: $\Delta W = (V_* - W K_*) (K_*^T C^{-1} K_* + \lambda I)^{-1} (C^{-1} K_*)^T$
3. **Model Stitching**: $S = (A^T A + \lambda I)^{-1} A^T B, \quad \hat{B} = A S$
4. **Surgical Neuron Pruning**: $W_{\text{pruned}} = W \odot (1 - m_{\text{knockout}})$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/10_xai_and_ai_analysis_methods/06_ai_brain_surgery_knowledge_editing_rome_memit_and_stitching.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def calc_rome_edit(w, k_star, v_star, cov_inv):
    residual = v_star - w @ k_star
    u = cov_inv @ k_star
    denom = (k_star * u).sum() + 1e-8
    delta_w = (residual.unsqueeze(1) @ u.unsqueeze(0)) / denom
    return delta_w, w + delta_w

def calc_memit_edit(w, keys, vals, cov_inv, reg=1e-4):
    residuals = vals - w @ keys
    u_mat = cov_inv @ keys
    gram = keys.t() @ u_mat + reg * torch.eye(keys.shape[1], dtype=keys.dtype, device=keys.device)
    coeff = torch.linalg.solve(gram.t(), residuals.t()).t()
    delta_w = coeff @ u_mat.t()
    return delta_w, w + delta_w

def calc_model_stitching(a, b, reg=1e-4):
    lhs = a.t() @ a + reg * torch.eye(a.shape[1], dtype=a.dtype, device=a.device)
    s = torch.linalg.solve(lhs, a.t() @ b)
    return s, a @ s

def calc_neuron_knockout(w, mask):
    return w * (1.0 - mask.unsqueeze(0))

In [ ]:
# Parity verification against golden reference tensors
r_delta, r_wnew = calc_rome_edit(weights["rome_w"], weights["rome_k_star"], weights["rome_v_star"], weights["rome_cov_inv"])
assert torch.max(torch.abs(r_delta - weights["rome_delta_ref"])) < 1e-5, "ROME delta mismatch"
assert torch.max(torch.abs(r_wnew - weights["rome_wnew_ref"])) < 1e-5, "ROME w_new mismatch"

m_delta, m_wnew = calc_memit_edit(weights["rome_w"], weights["memit_keys"], weights["memit_vals"], weights["rome_cov_inv"], reg=1e-4)
assert torch.max(torch.abs(m_delta - weights["memit_delta_ref"])) < 1e-5, "MEMIT delta mismatch"
assert torch.max(torch.abs(m_wnew - weights["memit_wnew_ref"])) < 1e-5, "MEMIT w_new mismatch"

s_mat, b_hat = calc_model_stitching(weights["stitch_a"], weights["stitch_b"], reg=1e-4)
assert torch.max(torch.abs(s_mat - weights["stitch_s_ref"])) < 1e-5, "Stitching matrix mismatch"
assert torch.max(torch.abs(b_hat - weights["stitch_bhat_ref"])) < 1e-5, "Stitched acts mismatch"

ko_w = calc_neuron_knockout(weights["rome_w"], weights["ko_mask"])
assert torch.max(torch.abs(ko_w - weights["ko_pruned_ref"])) < 1e-5, "Knockout mismatch"
print("All ROME, MEMIT, Stitching, and Knockout assertions passed!")

In [ ]:
# Benchmarking model surgery operators
t0 = time.perf_counter()
for _ in range(500):
    _ = calc_rome_edit(weights["rome_w"], weights["rome_k_star"], weights["rome_v_star"], weights["rome_cov_inv"])
    _ = calc_memit_edit(weights["rome_w"], weights["memit_keys"], weights["memit_vals"], weights["rome_cov_inv"], reg=1e-4)
    _ = calc_model_stitching(weights["stitch_a"], weights["stitch_b"], reg=1e-4)
    _ = calc_neuron_knockout(weights["rome_w"], weights["ko_mask"])
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Fused ROME + MEMIT + Stitching latency: {lat_us:.2f} us")

In [ ]:
# Failure Mode: ROME Exact Key Retrieval vs Unregularized Rank-1 Overwrite
# Proving that (W + Delta_W) @ k_* exactly equals v_*
k_star = weights["rome_k_star"]
v_star = weights["rome_v_star"]
w_edited = weights["rome_wnew_ref"]
v_retrieved = w_edited @ k_star
err_exact = float(torch.max(torch.abs(v_retrieved - v_star)))
print(f"Key retrieval absolute error: {err_exact:.2e}")
assert err_exact < 1e-5, "ROME guarantees exact factual retrieval!"
# Naive unregularized update v_* @ k_*^T fails to conserve prior knowledge
w_naive = v_star.unsqueeze(1) @ k_star.unsqueeze(0)
v_naive_ret = w_naive @ k_star
assert float(torch.norm(v_naive_ret - v_star)) > 1.0, "Naive outer product fails exact scale preservation!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **ROME** performs exact rank-one factual insertion in MLP weights while preserving historical key covariance.
2. **MEMIT** scales model editing to thousands of concurrent facts via multi-key least-squares projection.
3. **Model Stitching** verifies functional equivalence and modularity across distinct neural networks.
4. **Surgical Pruning** knocks out hallucinating or spurious channels with minimal parameter disruption.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [06_ai_brain_surgery_knowledge_editing_rome_memit_and_stitching.md](../../markdown/10_xai_and_ai_analysis_methods/06_ai_brain_surgery_knowledge_editing_rome_memit_and_stitching.md)
- **PyTorch 2.x Module:** [06_ai_brain_surgery_knowledge_editing_rome_memit_and_stitching.py](../../pytorch2/10_xai_and_ai_analysis_methods/06_ai_brain_surgery_knowledge_editing_rome_memit_and_stitching.py)
- **Functional JAX Module:** [06_ai_brain_surgery_knowledge_editing_rome_memit_and_stitching.py](../../jax/10_xai_and_ai_analysis_methods/06_ai_brain_surgery_knowledge_editing_rome_memit_and_stitching.py)
- **Native Rust Candle Track:** [06_ai_brain_surgery_knowledge_editing_rome_memit_and_stitching.rs](../../candle/10_xai_and_ai_analysis_methods/06_ai_brain_surgery_knowledge_editing_rome_memit_and_stitching.rs)
- **Hardware Kernel (CPU / WGSL):** [06_ai_brain_surgery_knowledge_editing_rome_memit_and_stitching.rs](../../kernels/10_xai_and_ai_analysis_methods/06_ai_brain_surgery_knowledge_editing_rome_memit_and_stitching.rs)
- **Golden SafeTensors:** [06_ai_brain_surgery_knowledge_editing_rome_memit_and_stitching.safetensors](../../weights/10_xai_and_ai_analysis_methods/06_ai_brain_surgery_knowledge_editing_rome_memit_and_stitching.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 05. Mechanistic Interpretability Probing Lens Patching SAE](05_mechanistic_interpretability_probing_lens_patching_and_sae.ipynb) | [Master Index](../../README.md#10_xai_and_ai_analysis_methods) | [Next: 07. Representation Similarity Metrics CKA SVCCA RSA](07_ai_analysis_representation_similarity_cka_svcca_rsa_and_intrinsic_dim.ipynb) &rarr;